## This notebook contains the RF/GBM baseline and the 25-seed evaluation of all five model conditions reported in the manuscript. This is the canonical source for Table 2 and all statistics cited in the paper.

In [1]:
# Imports and data loading

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, ttest_rel, t as t_dist
import json

BASE_DIR  = Path("..")
PROC_DIR  = BASE_DIR / "data" / "processed"

master = pd.read_csv(PROC_DIR / "training_table.csv")

all_cols    = master.columns.tolist()
morgan_cols = [c for c in all_cols if c.startswith('morgan_')]
morpho_cols = [c for c in all_cols if c.startswith('Cells_')
               or c.startswith('Nuclei_')
               or c.startswith('Cytoplasm_')]
ic50_cols   = [c for c in all_cols
               if c not in morgan_cols + morpho_cols + ['drug_name', 'Metadata_JCP2022']]

drug_names    = master['drug_name'].values
drug_index    = {name: i for i, name in enumerate(drug_names)}
morgan_lookup = master.set_index('drug_name')[morgan_cols].values.astype(np.float32)
morpho_lookup = master.set_index('drug_name')[morpho_cols].values.astype(np.float32)

ic50_long = master[['drug_name'] + ic50_cols].melt(
    id_vars='drug_name', var_name='cell_line', value_name='ln_ic50'
).dropna(subset=['ln_ic50']).reset_index(drop=True)

print(f"Drugs:       {len(drug_names)}")
print(f"Morgan cols: {len(morgan_cols)}")
print(f"Morpho cols: {len(morpho_cols)}")
print(f"IC50 pairs:  {len(ic50_long)}")

Drugs:       175
Morgan cols: 2048
Morpho cols: 3178
IC50 pairs:  149679


In [6]:
# Random Forest and Gradient Boosting baselines, seeds 0-24

from scipy import stats

SEEDS = list(range(25))
drug_mean_ic50 = ic50_long.groupby('drug_name')['ln_ic50'].mean()  # moved outside loop

rf_results  = {'r': [], 'rmse': []}
gbm_results = {'r': [], 'rmse': []}

for seed in SEEDS:
    np.random.seed(seed)
    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val
    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    test_drugs  = set(drug_names[shuffled_idx[n_train + n_val:]])
    train_idx = [drug_index[d] for d in train_drugs]
    test_idx  = [drug_index[d] for d in test_drugs]

    y_train = drug_mean_ic50.loc[list(train_drugs)].values
    y_test  = drug_mean_ic50.loc[list(test_drugs)].values
    X_train = morgan_lookup[train_idx]
    X_test  = morgan_lookup[test_idx]

    rf = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=seed)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    r_rf, _  = pearsonr(rf_pred, y_test)
    rmse_rf  = np.sqrt(mean_squared_error(y_test, rf_pred))
    rf_results['r'].append(float(r_rf))
    rf_results['rmse'].append(float(rmse_rf))

    gbm = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                     max_depth=4, random_state=seed)
    gbm.fit(X_train, y_train)
    gbm_pred = gbm.predict(X_test)
    r_gbm, _ = pearsonr(gbm_pred, y_test)
    rmse_gbm = np.sqrt(mean_squared_error(y_test, gbm_pred))
    gbm_results['r'].append(float(r_gbm))
    gbm_results['rmse'].append(float(rmse_gbm))

    print(f"[{seed+1}/25] seed={seed} | RF: R={r_rf:.4f} RMSE={rmse_rf:.4f} | "
          f"GBM: R={r_gbm:.4f} RMSE={rmse_gbm:.4f}")

print("\n" + "="*55)
print("mean ± sample std, n=25 seeds")
print("="*55)
for name, res in [("Random Forest", rf_results), ("Gradient Boosting", gbm_results)]:
    r_mean, r_std = np.mean(res['r']), np.std(res['r'], ddof=1)
    rmse_mean, rmse_std = np.mean(res['rmse']), np.std(res['rmse'], ddof=1)
    ci = stats.t.interval(0.95, len(res['r'])-1, loc=r_mean, scale=r_std/np.sqrt(len(res['r'])))
    print(f"{name:20s} | R: {r_mean:.3f} ± {r_std:.3f} [{ci[0]:.3f}, {ci[1]:.3f}] | "
          f"RMSE: {rmse_mean:.3f} ± {rmse_std:.3f}")

import json
with open('rf_gbm_25seed_results.json', 'w') as f:
    json.dump({'random_forest': rf_results, 'gradient_boosting': gbm_results}, f, indent=2)
print("\nSaved: rf_gbm_25seed_results.json")

[1/25] seed=0 | RF: R=0.3271 RMSE=1.8724 | GBM: R=0.3054 RMSE=1.8231
[2/25] seed=1 | RF: R=0.3445 RMSE=1.5289 | GBM: R=0.4230 RMSE=1.4974
[3/25] seed=2 | RF: R=0.6123 RMSE=1.9590 | GBM: R=0.6083 RMSE=1.9027
[4/25] seed=3 | RF: R=0.3661 RMSE=2.1398 | GBM: R=0.3621 RMSE=2.1633
[5/25] seed=4 | RF: R=0.1638 RMSE=1.7042 | GBM: R=0.0054 RMSE=1.8433
[6/25] seed=5 | RF: R=0.5211 RMSE=1.4603 | GBM: R=0.4182 RMSE=1.6404
[7/25] seed=6 | RF: R=0.4811 RMSE=1.5320 | GBM: R=0.4187 RMSE=1.5974
[8/25] seed=7 | RF: R=0.1752 RMSE=1.5925 | GBM: R=-0.0705 RMSE=1.7542
[9/25] seed=8 | RF: R=0.5425 RMSE=1.6958 | GBM: R=0.5275 RMSE=1.7116
[10/25] seed=9 | RF: R=0.5407 RMSE=2.3535 | GBM: R=0.5022 RMSE=2.3238
[11/25] seed=10 | RF: R=0.1742 RMSE=1.7338 | GBM: R=0.2478 RMSE=1.7723
[12/25] seed=11 | RF: R=0.0682 RMSE=1.5044 | GBM: R=0.0567 RMSE=1.5849
[13/25] seed=12 | RF: R=0.4745 RMSE=1.9436 | GBM: R=0.4618 RMSE=2.1702
[14/25] seed=13 | RF: R=0.6402 RMSE=1.8723 | GBM: R=0.6337 RMSE=1.8109
[15/25] seed=14 | RF: R=

In [2]:
# 25-seed evaluation of all five model conditions
# Expand from 5 seeds to 25, using seeds 0-24.
# Save results to results/multiseed_25_results.json (existing files untouched)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEEDS_25 = list(range(25))

class DrugResponseDataset(Dataset):
    def __init__(self, df, drug_index, morgan_sc, morpho_sc):
        self.drug_idx = torch.tensor(
            [drug_index[d] for d in df['drug_name']], dtype=torch.long)
        self.targets = torch.tensor(df['ln_ic50'].values, dtype=torch.float32)
        self.morgan  = torch.tensor(morgan_sc, dtype=torch.float32)
        self.morpho  = torch.tensor(morpho_sc, dtype=torch.float32)
    def __len__(self): return len(self.targets)
    def __getitem__(self, i):
        idx = self.drug_idx[i]
        return self.morgan[idx], self.morpho[idx], self.targets[i]

class MorganMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2048,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self, morgan, morpho): return self.net(morgan).squeeze(-1)

class MorphoMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3178,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self, morgan, morpho): return self.net(morpho).squeeze(-1)

class ConcatMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5226,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256),  nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self, morgan, morpho):
        return self.net(torch.cat([morgan, morpho], dim=1)).squeeze(-1)

class CrossAttentionFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178,
                 token_dim=64, num_tokens=16, num_heads=4,
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.num_tokens = num_tokens
        self.morgan_token_proj = nn.Linear(morgan_dim // num_tokens, token_dim)
        self.morpho_pad = (num_tokens - (morpho_dim % num_tokens)) % num_tokens
        self.morpho_token_proj = nn.Linear(
            (morpho_dim + self.morpho_pad) // num_tokens, token_dim)
        self.cross_attn_layers = nn.ModuleList([
            nn.MultiheadAttention(token_dim, num_heads,
                                  dropout=dropout, batch_first=True)
            for _ in range(num_layers)])
        self.layer_norms = nn.ModuleList(
            [nn.LayerNorm(token_dim) for _ in range(num_layers)])
        self.readout = nn.Sequential(
            nn.Linear(token_dim*num_tokens*2, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1))
    def tokenize(self, x, proj, n_tokens, pad=0):
        if pad > 0: x = F.pad(x, (0, pad))
        return proj(x.view(x.shape[0], n_tokens, -1))
    def forward(self, morgan, morpho):
        z_m = self.tokenize(morgan, self.morgan_token_proj, self.num_tokens)
        z_p = self.tokenize(morpho, self.morpho_token_proj,
                            self.num_tokens, pad=self.morpho_pad)
        attn_out = z_p
        for attn, norm in zip(self.cross_attn_layers, self.layer_norms):
            attended, _ = attn(attn_out, z_m, z_m)
            attn_out = norm(attn_out + attended)
        fused = torch.cat([attn_out.reshape(attn_out.shape[0], -1),
                           z_m.reshape(z_m.shape[0], -1)], dim=1)
        return self.readout(fused).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178,
                 embed_dim=256, dropout=0.3):
        super().__init__()
        self.morgan_proj = nn.Sequential(
            nn.Linear(morgan_dim, embed_dim), nn.LayerNorm(embed_dim),
            nn.ReLU(), nn.Dropout(dropout))
        self.morpho_proj = nn.Sequential(
            nn.Linear(morpho_dim, embed_dim), nn.LayerNorm(embed_dim),
            nn.ReLU(), nn.Dropout(dropout))
        self.gate = nn.Sequential(
            nn.Linear(embed_dim*2, embed_dim), nn.ReLU(),
            nn.Linear(embed_dim, 2), nn.Softmax(dim=1))
        self.readout = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1))
    def forward(self, morgan, morpho):
        z_m = self.morgan_proj(morgan)
        z_p = self.morpho_proj(morpho)
        gates = self.gate(torch.cat([z_m, z_p], dim=1))
        return self.readout(gates[:,0:1]*z_m + gates[:,1:2]*z_p).squeeze(-1)

def train_model(model, train_loader, val_loader,
                epochs=150, lr=1e-3, patience=20,
                weight_decay=1e-4, clip=False):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    best_val, best_state, patience_ctr = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for mb, mp, mt in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(mb, mp), mt)
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for mb, mp, mt in val_loader:
                val_loss += criterion(model(mb, mp), mt).item() * len(mt)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    return model

def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for mb, mp, mt in loader:
            preds.extend(model(mb, mp).numpy())
            targets.extend(mt.numpy())
    preds, targets = np.array(preds), np.array(targets)
    r, _ = pearsonr(preds, targets)
    rmse  = np.sqrt(((preds - targets)**2).mean())
    return float(r), float(rmse)

models_cfg = [
    ('structure_only',  MorganMLP,           False),
    ('morphology_only', MorphoMLP,            False),
    ('concatenation',   ConcatMLP,            False),
    ('cross_attention', CrossAttentionFusion, True),
    ('gated_fusion',    GatedFusion,          False),
]

results_25 = {name: {'r': [], 'rmse': []} for name, _, _ in models_cfg}

total_runs = len(SEEDS_25) * len(models_cfg)
completed  = 0

for seed in SEEDS_25:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_idx = np.random.permutation(len(drug_names))
    n_test  = int(len(drug_names) * 0.15)
    n_val   = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val

    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    val_drugs   = set(drug_names[shuffled_idx[n_train:n_train+n_val]])
    test_drugs  = set(drug_names[shuffled_idx[n_train+n_val:]])

    train_df = ic50_long[ic50_long['drug_name'].isin(train_drugs)].reset_index(drop=True)
    val_df   = ic50_long[ic50_long['drug_name'].isin(val_drugs)].reset_index(drop=True)
    test_df  = ic50_long[ic50_long['drug_name'].isin(test_drugs)].reset_index(drop=True)

    train_idx     = [drug_index[d] for d in train_drugs]
    morpho_scaler = StandardScaler()
    morpho_sc     = morpho_scaler.fit(
        morpho_lookup[train_idx]).transform(morpho_lookup)
    morgan_sc     = morgan_lookup

    train_ds = DrugResponseDataset(train_df, drug_index, morgan_sc, morpho_sc)
    val_ds   = DrugResponseDataset(val_df,   drug_index, morgan_sc, morpho_sc)
    test_ds  = DrugResponseDataset(test_df,  drug_index, morgan_sc, morpho_sc)

    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False)

    for name, ModelClass, clip in models_cfg:
        model = ModelClass()
        model = train_model(model, train_loader, val_loader, clip=clip)
        r, rmse = evaluate(model, test_loader)
        results_25[name]['r'].append(r)
        results_25[name]['rmse'].append(rmse)
        completed += 1
        print(f"[{completed}/{total_runs}] seed={seed} {name}: "
              f"R={r:.4f} RMSE={rmse:.4f}")

print("\n" + "="*65)
print("mean ± sample std, n=25 seeds")
print("="*65)
for name in results_25:
    r_arr     = np.array(results_25[name]['r'])
    rmse_arr  = np.array(results_25[name]['rmse'])
    r_mean    = r_arr.mean()
    r_std     = r_arr.std(ddof=1)
    rmse_mean = rmse_arr.mean()
    rmse_std  = rmse_arr.std(ddof=1)
    n         = len(r_arr)
    t_crit    = t_dist.ppf(0.975, df=n-1)
    ci_lo     = r_mean - t_crit * r_std / np.sqrt(n)
    ci_hi     = r_mean + t_crit * r_std / np.sqrt(n)
    print(f"{name:20s} | R: {r_mean:.3f} ± {r_std:.3f} "
          f"[{ci_lo:.3f}, {ci_hi:.3f}] | RMSE: {rmse_mean:.3f} ± {rmse_std:.3f}")

print("\nPaired t-tests (25 seeds):")
pairs = [
    ('structure_only',  'morphology_only', 'structure vs morphology'),
    ('morphology_only', 'concatenation',   'morphology vs concatenation'),
    ('concatenation',   'cross_attention', 'concatenation vs cross-attention'),
    ('concatenation',   'gated_fusion',    'concatenation vs gated fusion'),
]
for a, b, label in pairs:
    t_stat, p_val = ttest_rel(
        results_25[b]['r'], results_25[a]['r'])
    print(f"  {label}: t={t_stat:.2f}, p={p_val:.4f}")

with open(BASE_DIR / "results" / "multiseed_25_results.json", "w") as f:
    json.dump(results_25, f, indent=2)
print("\nSaved: multiseed_25_results.json")

[1/125] seed=0 structure_only: R=0.1527 RMSE=2.4421
[2/125] seed=0 morphology_only: R=0.5708 RMSE=1.9076
[3/125] seed=0 concatenation: R=0.5827 RMSE=1.8682
[4/125] seed=0 cross_attention: R=0.6425 RMSE=1.7427
[5/125] seed=0 gated_fusion: R=0.5369 RMSE=1.9128
[6/125] seed=1 structure_only: R=0.3011 RMSE=2.2309
[7/125] seed=1 morphology_only: R=0.4132 RMSE=1.9866
[8/125] seed=1 concatenation: R=0.4508 RMSE=1.9347
[9/125] seed=1 cross_attention: R=0.4694 RMSE=1.9349
[10/125] seed=1 gated_fusion: R=0.4805 RMSE=1.8896
[11/125] seed=2 structure_only: R=0.4754 RMSE=2.5907
[12/125] seed=2 morphology_only: R=0.7014 RMSE=2.0301
[13/125] seed=2 concatenation: R=0.6903 RMSE=2.0577
[14/125] seed=2 cross_attention: R=0.6924 RMSE=2.0557
[15/125] seed=2 gated_fusion: R=0.5884 RMSE=2.3159
[16/125] seed=3 structure_only: R=0.3373 RMSE=2.4464
[17/125] seed=3 morphology_only: R=0.6574 RMSE=1.9946
[18/125] seed=3 concatenation: R=0.6823 RMSE=1.9200
[19/125] seed=3 cross_attention: R=0.7269 RMSE=1.8234
[20/